# Shortest Paths with Edge Capacities

This notebook demonstrates how to route multiple paths through a network graph while respecting edge capacities. Each edge has a maximum capacity, and when a path uses an edge, its remaining capacity decreases.

**Adapted from topologicpy to use topologic_fast**

Key concepts:
1. Create a network graph with vertices and edges
2. Assign capacity values to edges
3. Find multiple shortest paths that respect capacity constraints
4. Visualize the results with Plotly

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
import random
import math
from collections import defaultdict

## 1. Create a Grid Network

Since topologic_fast doesn't have DXF import, we'll create a grid network programmatically.
This represents a typical routing network (pipes, cables, etc.)

In [ ]:
# Create a grid network
# Grid parameters
grid_width = 8  # nodes horizontally
grid_height = 4  # nodes vertically
spacing = 1.0   # distance between nodes

# Create vertices in a grid pattern
vertices = []
vertex_map = {}  # (i, j) -> vertex index

for j in range(grid_height):
    for i in range(grid_width):
        v = tf.Vertex.ByCoordinates(i * spacing, j * spacing, 0.0)
        vertex_map[(i, j)] = len(vertices)
        vertices.append(v)

print(f"Created {len(vertices)} vertices in a {grid_width}x{grid_height} grid")

In [ ]:
# Create edges connecting adjacent vertices (horizontal and vertical)
edges = []

for j in range(grid_height):
    for i in range(grid_width):
        # Horizontal edge (to the right)
        if i < grid_width - 1:
            v1 = vertices[vertex_map[(i, j)]]
            v2 = vertices[vertex_map[(i + 1, j)]]
            edge = tf.Edge.ByStartVertexEndVertex(v1, v2)
            edges.append(edge)
        
        # Vertical edge (upward)
        if j < grid_height - 1:
            v1 = vertices[vertex_map[(i, j)]]
            v2 = vertices[vertex_map[(i, j + 1)]]
            edge = tf.Edge.ByStartVertexEndVertex(v1, v2)
            edges.append(edge)

print(f"Created {len(edges)} edges")

In [ ]:
# Create graph from vertices and edges
graph = tf.Graph.ByVerticesEdges(vertices, edges)

print(f"Graph created with:")
print(f"  Vertices: {graph.Order()}")
print(f"  Edges: {graph.Size()}")

## 2. Assign Capacities to Edges

Each edge gets a random capacity between 2 and 6. We'll track capacities in a dictionary since topologic_fast edge dictionaries work differently.

**Note:** topologic_fast doesn't have the same Dictionary system as topologicpy, so we use Python dictionaries keyed by edge index.

In [ ]:
# Get edges from the graph
graph_edges = graph.Edges()

# Assign random capacities to each edge
# We use edge midpoint as a key since edges don't have persistent IDs
edge_capacities = {}
edge_midpoints = {}

for i, edge in enumerate(graph_edges):
    capacity = random.randint(2, 6)
    midpoint = edge.Midpoint()
    # Create a hashable key from midpoint coordinates
    key = (round(midpoint[0], 4), round(midpoint[1], 4), round(midpoint[2], 4))
    edge_capacities[key] = capacity
    edge_midpoints[i] = key

print(f"Assigned capacities to {len(edge_capacities)} edges")
print(f"Capacity range: {min(edge_capacities.values())} to {max(edge_capacities.values())}")

## 3. Define Start and End Points

We'll route paths from the bottom-left corner to the top-right corner of the grid.

In [ ]:
# Start vertex: bottom-left (0, 0)
# End vertex: top-right (grid_width-1, grid_height-1)
start_vertex = vertices[vertex_map[(0, 0)]]
end_vertex = vertices[vertex_map[(grid_width - 1, grid_height - 1)]]

print(f"Start: {start_vertex.Coordinates()}")
print(f"End: {end_vertex.Coordinates()}")

## 4. Implement Capacity-Aware Shortest Path

We implement a custom shortest path algorithm that:
1. Only uses edges with remaining capacity > 0
2. Reduces edge capacity after each path is found
3. Uses BFS to find shortest paths

**Note:** topologic_fast's Graph.Path() doesn't support edge filtering, so we implement our own algorithm.

In [ ]:
def get_edge_key(edge):
    """Get a hashable key for an edge based on its midpoint."""
    midpoint = edge.Midpoint()
    return (round(midpoint[0], 4), round(midpoint[1], 4), round(midpoint[2], 4))

def get_vertex_key(vertex):
    """Get a hashable key for a vertex based on its coordinates."""
    coords = vertex.Coordinates()
    return (round(coords[0], 4), round(coords[1], 4), round(coords[2], 4))

def build_adjacency_with_capacity(graph, edge_capacities):
    """Build adjacency list only including edges with remaining capacity."""
    adjacency = defaultdict(list)  # vertex_key -> [(neighbor_key, edge_key)]
    
    graph_edges = graph.Edges()
    for edge in graph_edges:
        edge_key = get_edge_key(edge)
        capacity = edge_capacities.get(edge_key, 0)
        
        if capacity > 0:
            start, end = edge.Vertices()
            start_key = get_vertex_key(start)
            end_key = get_vertex_key(end)
            
            # Bidirectional
            adjacency[start_key].append((end_key, edge_key))
            adjacency[end_key].append((start_key, edge_key))
    
    return adjacency

def find_shortest_path_bfs(graph, start_vertex, end_vertex, edge_capacities):
    """Find shortest path using BFS, respecting edge capacities."""
    from collections import deque
    
    adjacency = build_adjacency_with_capacity(graph, edge_capacities)
    
    start_key = get_vertex_key(start_vertex)
    end_key = get_vertex_key(end_vertex)
    
    # BFS
    queue = deque([(start_key, [start_key], [])])  # (current, vertex_path, edge_path)
    visited = {start_key}
    
    while queue:
        current, vertex_path, edge_path = queue.popleft()
        
        if current == end_key:
            return vertex_path, edge_path
        
        for neighbor, edge_key in adjacency.get(current, []):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, vertex_path + [neighbor], edge_path + [edge_key]))
    
    return None, None  # No path found

print("Shortest path functions defined.")

## 5. Find Multiple Paths with Capacity Constraints

In [ ]:
# Number of paths to route
n_paths = 7

# Store all found paths
found_paths = []  # List of (vertex_keys, edge_keys, path_length)

# Make a copy of capacities that we'll modify
current_capacities = dict(edge_capacities)

for i in range(n_paths):
    vertex_path, edge_path = find_shortest_path_bfs(graph, start_vertex, end_vertex, current_capacities)
    
    if vertex_path is None:
        print(f"Path {i + 1}: No path found (all routes at capacity)")
        break
    
    # Calculate path length
    path_length = len(edge_path) * spacing
    found_paths.append((vertex_path, edge_path, path_length))
    
    print(f"Path {i + 1}: Length = {path_length:.2f} m, using {len(edge_path)} edges")
    
    # Reduce capacity of used edges
    for edge_key in edge_path:
        if edge_key in current_capacities:
            current_capacities[edge_key] -= 1

print(f"\nSuccessfully routed {len(found_paths)} paths")

## 6. Visualize the Network and Paths with Plotly

In [ ]:
def visualize_network_2d(graph, edge_capacities, found_paths):
    """Create a 2D visualization of the network and paths."""
    fig = go.Figure()
    
    graph_edges = graph.Edges()
    graph_vertices = graph.Vertices()
    
    # Draw all edges with capacity labels
    for edge in graph_edges:
        start, end = edge.Vertices()
        start_coords = start.Coordinates()
        end_coords = end.Coordinates()
        
        edge_key = get_edge_key(edge)
        capacity = edge_capacities.get(edge_key, 0)
        
        # Color based on remaining capacity
        if capacity == 0:
            color = 'rgba(200, 200, 200, 0.3)'  # Gray for exhausted
        else:
            color = 'rgba(100, 100, 100, 0.5)'  # Dark gray for available
        
        fig.add_trace(go.Scatter(
            x=[start_coords[0], end_coords[0]],
            y=[start_coords[1], end_coords[1]],
            mode='lines',
            line=dict(color=color, width=3),
            showlegend=False,
            hoverinfo='skip'
        ))
    
    # Draw capacity labels at edge midpoints
    midpoint_x = []
    midpoint_y = []
    capacity_labels = []
    
    for edge in graph_edges:
        midpoint = edge.Midpoint()
        edge_key = get_edge_key(edge)
        capacity = edge_capacities.get(edge_key, 0)
        
        midpoint_x.append(midpoint[0])
        midpoint_y.append(midpoint[1])
        capacity_labels.append(str(capacity))
    
    fig.add_trace(go.Scatter(
        x=midpoint_x,
        y=midpoint_y,
        mode='text',
        text=capacity_labels,
        textfont=dict(size=10, color='darkblue'),
        showlegend=False,
        hoverinfo='skip'
    ))
    
    # Draw found paths with different colors
    path_colors = [
        'red', 'green', 'blue', 'orange', 'purple', 'cyan', 'magenta',
        'brown', 'pink', 'olive'
    ]
    
    for idx, (vertex_path, edge_path, path_length) in enumerate(found_paths):
        color = path_colors[idx % len(path_colors)]
        z_offset = (idx + 1) * 0.1  # Slight vertical offset for visibility
        
        # Draw path segments
        path_x = [vk[0] for vk in vertex_path]
        path_y = [vk[1] for vk in vertex_path]
        
        fig.add_trace(go.Scatter(
            x=path_x,
            y=path_y,
            mode='lines+markers',
            line=dict(color=color, width=4),
            marker=dict(size=8, color=color),
            name=f'Path {idx + 1} ({path_length:.2f}m)',
            hoverinfo='name'
        ))
    
    # Draw all vertices
    vertex_x = [v.Coordinates()[0] for v in graph_vertices]
    vertex_y = [v.Coordinates()[1] for v in graph_vertices]
    
    fig.add_trace(go.Scatter(
        x=vertex_x,
        y=vertex_y,
        mode='markers',
        marker=dict(size=12, color='white', line=dict(color='black', width=2)),
        name='Network Nodes',
        hoverinfo='skip'
    ))
    
    # Mark start and end
    start_coords = start_vertex.Coordinates()
    end_coords = end_vertex.Coordinates()
    
    fig.add_trace(go.Scatter(
        x=[start_coords[0]],
        y=[start_coords[1]],
        mode='markers+text',
        marker=dict(size=20, color='green', symbol='star'),
        text=['START'],
        textposition='bottom center',
        name='Start',
        showlegend=True
    ))
    
    fig.add_trace(go.Scatter(
        x=[end_coords[0]],
        y=[end_coords[1]],
        mode='markers+text',
        marker=dict(size=20, color='red', symbol='star'),
        text=['END'],
        textposition='top center',
        name='End',
        showlegend=True
    ))
    
    fig.update_layout(
        title='Network with Capacity-Constrained Shortest Paths',
        xaxis=dict(
            title='X',
            scaleanchor='y',
            scaleratio=1,
            range=[-0.5, grid_width * spacing - 0.5]
        ),
        yaxis=dict(
            title='Y',
            range=[-0.5, grid_height * spacing - 0.5]
        ),
        width=1000,
        height=500,
        showlegend=True,
        legend=dict(x=1.02, y=1)
    )
    
    return fig

fig = visualize_network_2d(graph, edge_capacities, found_paths)
fig.show()

## 7. Visualize Remaining Capacities

In [ ]:
def visualize_remaining_capacities(graph, original_capacities, current_capacities):
    """Visualize edges colored by remaining capacity."""
    fig = go.Figure()
    
    graph_edges = graph.Edges()
    
    # Color scale: red (0) -> yellow (mid) -> green (full)
    max_capacity = max(original_capacities.values())
    
    for edge in graph_edges:
        start, end = edge.Vertices()
        start_coords = start.Coordinates()
        end_coords = end.Coordinates()
        
        edge_key = get_edge_key(edge)
        remaining = current_capacities.get(edge_key, 0)
        original = original_capacities.get(edge_key, 1)
        
        # Calculate color based on remaining capacity ratio
        ratio = remaining / original if original > 0 else 0
        
        if ratio == 0:
            color = 'red'
        elif ratio < 0.5:
            color = 'orange'
        elif ratio < 1.0:
            color = 'yellow'
        else:
            color = 'green'
        
        width = 2 + remaining * 2  # Thicker edges have more capacity
        
        fig.add_trace(go.Scatter(
            x=[start_coords[0], end_coords[0]],
            y=[start_coords[1], end_coords[1]],
            mode='lines',
            line=dict(color=color, width=width),
            showlegend=False,
            hovertext=f'Remaining: {remaining}/{original}',
            hoverinfo='text'
        ))
    
    # Add legend entries
    for color, label in [('green', 'Full capacity'), ('yellow', '50-99%'), 
                         ('orange', '1-49%'), ('red', 'Exhausted')]:
        fig.add_trace(go.Scatter(
            x=[None], y=[None],
            mode='lines',
            line=dict(color=color, width=4),
            name=label
        ))
    
    fig.update_layout(
        title='Remaining Edge Capacities After Routing',
        xaxis=dict(title='X', scaleanchor='y', scaleratio=1),
        yaxis=dict(title='Y'),
        width=900,
        height=450,
        showlegend=True
    )
    
    return fig

fig_capacity = visualize_remaining_capacities(graph, edge_capacities, current_capacities)
fig_capacity.show()

## 8. Path Statistics

In [ ]:
# Calculate statistics
print("Path Statistics")
print("=" * 50)

total_length = 0
for idx, (vertex_path, edge_path, path_length) in enumerate(found_paths):
    print(f"Path {idx + 1}: {path_length:.2f} m ({len(edge_path)} edges)")
    total_length += path_length

print(f"\nTotal routing length: {total_length:.2f} m")
print(f"Average path length: {total_length / len(found_paths):.2f} m")

# Count exhausted edges
exhausted = sum(1 for c in current_capacities.values() if c == 0)
total_edges = len(current_capacities)
print(f"\nExhausted edges: {exhausted}/{total_edges} ({100*exhausted/total_edges:.1f}%)")

## 9. 3D Visualization with Stacked Paths

In [ ]:
def visualize_3d_paths(graph, found_paths):
    """Create 3D visualization with paths stacked vertically."""
    fig = go.Figure()
    
    graph_edges = graph.Edges()
    
    # Draw base network at z=0
    for edge in graph_edges:
        start, end = edge.Vertices()
        start_coords = start.Coordinates()
        end_coords = end.Coordinates()
        
        fig.add_trace(go.Scatter3d(
            x=[start_coords[0], end_coords[0]],
            y=[start_coords[1], end_coords[1]],
            z=[0, 0],
            mode='lines',
            line=dict(color='lightgray', width=2),
            showlegend=False,
            hoverinfo='skip'
        ))
    
    # Draw paths at increasing z levels
    path_colors = [
        'red', 'green', 'blue', 'orange', 'purple', 'cyan', 'magenta'
    ]
    
    z_spacing = 0.3
    
    for idx, (vertex_path, edge_path, path_length) in enumerate(found_paths):
        color = path_colors[idx % len(path_colors)]
        z_level = (idx + 1) * z_spacing
        
        path_x = [vk[0] for vk in vertex_path]
        path_y = [vk[1] for vk in vertex_path]
        path_z = [z_level] * len(vertex_path)
        
        fig.add_trace(go.Scatter3d(
            x=path_x,
            y=path_y,
            z=path_z,
            mode='lines+markers',
            line=dict(color=color, width=6),
            marker=dict(size=4, color=color),
            name=f'Path {idx + 1} ({path_length:.2f}m)'
        ))
    
    fig.update_layout(
        title='3D View: Stacked Routing Paths',
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Path Layer',
            aspectmode='manual',
            aspectratio=dict(x=2, y=1, z=1),
            camera=dict(eye=dict(x=1.5, y=-1.5, z=1))
        ),
        width=1000,
        height=600
    )
    
    return fig

fig_3d = visualize_3d_paths(graph, found_paths)
fig_3d.show()

## Summary

This notebook demonstrated:

1. **Grid Network Creation** - Built a graph programmatically using `tf.Vertex`, `tf.Edge`, and `tf.Graph`
2. **Capacity Management** - Assigned and tracked edge capacities using Python dictionaries
3. **Capacity-Aware Routing** - Implemented BFS-based shortest path that respects capacity constraints
4. **Visualization** - Created 2D and 3D visualizations with Plotly

### Key Differences from topologicpy:

- **No DXF Import**: topologic_fast doesn't have `Topology.ByDXFPath()`, so we create networks programmatically
- **No Edge Filter in Path**: topologic_fast's `Graph.Path()` doesn't support edge filtering, so we implemented custom BFS
- **Dictionary Handling**: Edge dictionaries work differently; we use Python dicts with coordinate-based keys
- **No Topology.Show()**: We use Plotly directly for visualization

### Applications:
- **MEP Routing**: Pipes, ducts, cables in buildings
- **Network Design**: Telecommunications, transportation
- **Resource Allocation**: Any network with limited edge capacity